# BSRNN 을 `torch.compile` 할 수 있나 — LSTM 이 걸리는 지점

[wesep](https://github.com/wenet-e2e/wesep) 의 [BSRNN](../wesep/models/bsrnn.py) 학습을 컴파일해서 돌릴 수 있는지 조사함.

**계기** — [examples/librimix/tse/v2/run.sh](../examples/librimix/tse/v2/run.sh) 의 stage 3 을
`--config confs/bsrnn_ecapa_FiLM.yaml --debug true` 로 디버깅하던 중,
컴파일을 얹을 수 있는지와 인자를 어디에 둘지가 문제가 됨.

이 노트북이 답하는 것:

| # | 물음 | 답 |
|---|---|---|
| 1 | BSRNN 이 컴파일되나 | 됨. 단 **graph break 5개**, 그중 3개가 LSTM |
| 2 | 얼마나 빨라지나 | **1.05×** — 연산 대부분인 LSTM 이 eager 로 빠져서 |
| 3 | 등록 발화 길이가 배치마다 다른데 괜찮나 | 재컴파일 2회 뒤 동적 축으로 안정됨 |
| 4 | **LSTM 도 컴파일이 되나** | `torch._dynamo.config.allow_rnn = True` 면 **됨.** 기본값은 `False` |
| 5 | `model.compile()` 이면 `_orig_mod.` 가 안 붙나 | **안 붙음** |
| 6 | wesep 쪽에 걸리는 함정 | [checkpoint.py](../wesep/utils/checkpoint.py) 의 `strict=False` |

| 항목 | 내용 |
|---|---|
| **커널** | **`wesep2`** — py3.9 · torch 2.7.1+cu128 |
| **GPU** | RTX 3090 1장 (`CUDA_VISIBLE_DEVICES=3`) |
| **건드리지 않은 것** | **저장소 파일을 하나도 고치지 않음.** 전부 이 노트북 안에서 모델을 새로 만들어 잼.<br>학습·체크포인트도 건드리지 않음 |
| **측정 조건** | 혼합 `(b=8, t=48000)` · 등록 fbank `(b=8, t=598, f=80)`.<br>`b=8` 은 [confs/bsrnn_ecapa_FiLM.yaml](../examples/librimix/tse/v2/confs/bsrnn_ecapa_FiLM.yaml) 의 `batch_size: 4` 를<br>[tse_collate_fn](../wesep/dataset/dataset.py) 이 화자 2명 몫으로 복제한 결과임 |
| **재는 것** | `ms/step` = forward + backward + `zero_grad` 1회의 벽시계 시간, ms |


In [1]:
import os, sys, time, platform, tempfile
from pathlib import Path

# Inductor 는 컴파일 결과를 디스크에 캐시함(기본 /tmp/torchinductor_<user>).
# 그대로 두면 두 번째 실행부터 "첫 스텝" 이 콜드가 아니게 되어 이 노트북의 콜드 수치가 거짓이 됨.
# 매 실행 새 폴더를 주어 항상 콜드에서 재도록 함 — torch 를 import 하기 전에 설정해야 함.
os.environ["TORCHINDUCTOR_CACHE_DIR"] = tempfile.mkdtemp(prefix="inductor_nb_")

import pandas as pd
import torch
import torch._dynamo as dynamo


def find_wesep_root(start: Path) -> Path:
    """`wesep/__init__.py` 를 담고 있는 폴더를 위로 올라가며 찾음 — 그것이 wesep 클론 루트임."""
    for d in (start, *start.parents):
        if (d / "wesep" / "__init__.py").is_file():
            return d
    raise FileNotFoundError(f"{start} 위쪽에 wesep 클론 루트가 없음")


WESEP = find_wesep_root(Path.cwd())
if str(WESEP) not in sys.path:
    sys.path.insert(0, str(WESEP))

from wesep.models import get_model

print("python  ", platform.python_version())
print("torch   ", torch.__version__, "| cuda", torch.version.cuda)
print("gpu     ", torch.cuda.get_device_name(0))
print("wesep   ", WESEP)
print("inductor cache", os.environ["TORCHINDUCTOR_CACHE_DIR"], "(매 실행 새로 만듦 - 콜드 보장)")

/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/kaldiio/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/s3prl/upstream/byol_s/byol_a/common.py:20: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("sox_io")
ESPnet is not installed, cannot use espnet_hubert upstream


/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


python   3.9.25
torch    2.7.1+cu128 | cuda 12.8
gpu      NVIDIA GeForce RTX 3090
wesep    /workspace/git_clone/SD-FiLM/wesep
inductor cache /tmp/inductor_nb_ya6ozzol (매 실행 새로 만듦 - 콜드 보장)


In [2]:
# 측정 대상 — confs/bsrnn_ecapa_FiLM.yaml 의 model_args.tse_model 을 그대로 옮긴 것.
# spk_model_init 만 False 로 둠: 사전학습 화자 인코더 가중치는 속도·graph break 와 무관하고,
# 파일을 읽지 않아야 이 노트북이 어디서든 돌기 때문임.
ARGS = dict(
    sr=16000, win=512, stride=128, feature_dim=128, num_repeat=6,
    spk_fuse_type="FiLM", use_spk_transform=False, multi_fuse=False,
    joint_training=True, spk_model="ECAPA_TDNN_GLOB_c512", spk_model_init=False,
    spk_args=dict(feat_dim=80, embed_dim=192, pooling_func="ASTP"),
    spk_emb_dim=192, spk_model_freeze=True, spk_feat=True,
    feat_type="consistent", multi_task=False, spksInTrain=251,
)
B, T, TE, F = 8, 48000, 598, 80      # 혼합 (b=8, t=48000) · 등록 fbank (b=8, t=598, f=80)

torch.manual_seed(0)


def build_bsrnn():
    m = get_model("BSRNN")(**ARGS).cuda()
    m.train()
    return m


def step(m, x, e):
    """forward + backward + zero_grad 1회. 손실은 속도만 재므로 아무래도 좋음."""
    m(x, e)[0].pow(2).mean().backward()
    m.zero_grad(set_to_none=True)


def bench(m, n=10, te=TE, warm=3):
    x = torch.randn(B, T, device="cuda")
    e = torch.randn(B, te, F, device="cuda")
    for _ in range(warm):
        step(m, x, e)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n):
        step(m, x, e)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n * 1000


rows = []                            # 절마다 여기에 쌓고 마지막 셀에서 한 번에 표로 봄
m = build_bsrnn()
print("파라미터 수: {:.2f} M".format(sum(p.numel() for p in m.parameters()) / 1e6))

파라미터 수: 27.64 M


## 1. BSRNN 이 컴파일되나 — graph break 원문

`torch._dynamo.explain()` 은 **백엔드를 돌리지 않고** 추적만 해 그래프가 몇 조각으로 쪼개지는지 알려줌.
break 이유는 **원문이 근거**이므로 표로 뭉개지 않고 `print` 로 그대로 둠.


In [3]:
dynamo.reset()
m_ex = build_bsrnn()
x = torch.randn(B, T, device="cuda")
e = torch.randn(B, TE, F, device="cuda")

exp = dynamo.explain(m_ex)(x, e)
print("graph_count      ", exp.graph_count)
print("graph_break_count", exp.graph_break_count)
print("op_count         ", exp.op_count)
print()
for i, b in enumerate(exp.break_reasons, 1):
    print(f"--- BREAK {i} ---")
    print(str(b.reason).strip()[:400])
    print()

del m_ex
torch.cuda.empty_cache()

graph_count       6
graph_break_count 5
op_count          1087

--- BREAK 1 ---
TorchDynamo purposely graph breaks on RNN, GRU, LSTMs

--- BREAK 2 ---
Data dependent operator
  Explanation: Operator `aten._local_scalar_dense.default` has a non-Tensor output whose value is dependent on the data of Tensor inputs.
  Hint: Enable tracing of data-dependent output operators with `torch._dynamo.config.capture_scalar_outputs = True`

  Developer debug context: aten._local_scalar_dense.default

--- BREAK 3 ---
TorchDynamo purposely graph breaks on RNN, GRU, LSTMs

--- BREAK 4 ---
TorchDynamo purposely graph breaks on RNN, GRU, LSTMs



**5개 중 3개가 `TorchDynamo purposely graph breaks on RNN, GRU, LSTMs` 임.**

[bsrnn.py](../wesep/models/bsrnn.py) 의 연산 대부분이 `ResRNN` 의 `nn.LSTM` 인데
(`num_repeat=6` × `band_rnn`·`band_comm` = **LSTM 12개**),
그 12개가 전부 eager 로 빠지면 Inductor 가 융합할 데는 사이사이의
`GroupNorm` · `Linear` · `Conv1d` 뿐임. 다음 절의 1.05× 가 그 몫임.


## 2. eager 대 compile — 실제 속도

`mode="reduce-overhead"` 는 CUDA Graphs 를 노리는 모드임.
여기서는 **CUDA Graphs 가 적용되지 않는데**, [bsrnn.py:313](../wesep/models/bsrnn.py#L313) 과
[:386](../wesep/models/bsrnn.py#L386) 이 `torch.hann_window(self.win)` 를 **CPU 에서 만들어 매 forward 마다 `.to(device)`** 하기 때문임.
Inductor 가 `skipping cudagraphs due to cpu device (hann_window)` 를 찍음.


In [4]:
# eager
dynamo.reset()
m_eager = build_bsrnn()
ms_eager = bench(m_eager)
rows.append({"case": "eager", "first_step_s": None, "ms_per_step": round(ms_eager, 1),
             "note": "기준"})
print("eager", round(ms_eager, 1), "ms/step", flush=True)
del m_eager
torch.cuda.empty_cache()

eager 476.9 ms/step


In [5]:
for case, kw in [("compile 기본", {}), ("compile reduce-overhead", dict(mode="reduce-overhead"))]:
    dynamo.reset()
    dynamo.config.cache_size_limit = 16
    m_c = build_bsrnn()
    cm = torch.compile(m_c, **kw)
    x = torch.randn(B, T, device="cuda")
    e = torch.randn(B, TE, F, device="cuda")

    torch.cuda.synchronize()
    t0 = time.perf_counter()
    step(cm, x, e)                       # 첫 스텝 = 컴파일이 실제로 일어나는 곳
    torch.cuda.synchronize()
    first = time.perf_counter() - t0

    ms = bench(cm)
    rows.append({"case": case, "first_step_s": round(first, 1), "ms_per_step": round(ms, 1),
                 "note": f"{ms_eager / ms:.2f}x"})
    print(case, "first", round(first, 1), "s | steady", round(ms, 1), "ms/step", flush=True)
    del m_c, cm
    torch.cuda.empty_cache()

display(pd.DataFrame(rows))

/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/torch/_inductor/lowering.py:1917: UserWarning: Torchinductor does not support code generation for complex operators. Performance may be worse than eager.
  warnings.warn(


/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/torch/_inductor/compile_fx.py:236: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


compile 기본 first 97.5 s | steady 456.3 ms/step


skipping cudagraphs due to skipping cudagraphs due to cpu device (hann_window). Found from : 
   File "/workspace/git_clone/SD-FiLM/wesep/wesep/models/bsrnn.py", line 313, in forward
    window=torch.hann_window(self.win).to(wav_input.device).type(



skipping cudagraphs due to skipping cudagraphs due to cpu device (primals_7). Found from : 
   File "/workspace/git_clone/SD-FiLM/wesep/wesep/models/bsrnn.py", line 143, in forward
    x = x.view(batch_size * nch, self.nband * self.feature_dim, -1)



...

/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/torch/cuda/graphs.py:84: UserWarning: The CUDA Graph is empty. This usually means that the graph was attempted to be captured on wrong device or stream. (Triggered internally at /pytorch/aten/src/ATen/cuda/CUDAGraph.cpp:175.)
  super().capture_end()


/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/torch/cuda/graphs.py:84: UserWarning: The CUDA Graph is empty. This usually means that the graph was attempted to be captured on wrong device or stream. (Triggered internally at /pytorch/aten/src/ATen/cuda/CUDAGraph.cpp:175.)
  super().capture_end()


/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/torch/cuda/graphs.py:84: UserWarning: The CUDA Graph is empty. This usually means that the graph was attempted to be captured on wrong device or stream. (Triggered internally at /pytorch/aten/src/ATen/cuda/CUDAGraph.cpp:175.)
  super().capture_end()


compile reduce-overhead first 79.7 s | steady 452.9 ms/step


,case,first_step_s,ms_per_step,note
0,eager,NaN,476.9,기준
1,compile 기본,97.5,456.3,1.05x
2,compile reduce-overhead,79.7,452.9,1.05x


## 3. 등록 발화 길이가 배치마다 다른데 괜찮나

[tse_collate_fn](../wesep/dataset/dataset.py) 은 배치 안 등록 발화들을 **가장 짧은 것에 맞춰 자름**(`mode="min"`).
따라서 `(b, t, f=80)` 의 `t` 가 **배치마다 다름** — `torch.compile` 이 입력 모양에 guard 를 걸므로 재컴파일이 일어남.

길이를 여섯 번 바꿔 가며 스텝 시간을 재면, 재컴파일이 몇 번에 그치는지 보임.


In [6]:
dynamo.reset()
dynamo.config.cache_size_limit = 16
m_dyn = build_bsrnn()
cm = torch.compile(m_dyn)

len_rows = []
for L in [598, 512, 701, 455, 623, 380]:
    x = torch.randn(B, T, device="cuda")
    e = torch.randn(B, L, F, device="cuda")
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    step(cm, x, e)
    torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    len_rows.append({"enroll_len": L, "step_s": round(dt, 2),
                     "재컴파일": "예" if dt > 2 else "아니오"})
    print("enroll_len", L, "->", round(dt, 2), "s", flush=True)

display(pd.DataFrame(len_rows))
del m_dyn, cm
torch.cuda.empty_cache()

enroll_len 598 -> 9.88 s


enroll_len 512 -> 33.56 s


enroll_len 701 -> 0.48 s


enroll_len 455 -> 0.47 s


enroll_len 623 -> 0.47 s


enroll_len 380 -> 0.47 s


,enroll_len,step_s,재컴파일
0,598,9.88,예
1,512,33.56,예
2,701,0.48,아니오
3,455,0.47,아니오
4,623,0.47,아니오
5,380,0.47,아니오


**앞 2회만 재컴파일되고 그 뒤로는 안 함.**

dynamo 가 같은 축에서 두 번째 모양 변화를 보면 그 축을 **동적(dynamic)** 으로 표시하고
길이에 무관한 커널 하나를 만들기 때문임. 등록 발화 길이는 걱정거리가 아님.


## 4. LSTM 이 왜 빠지나 — dynamo 소스

추측하지 않고 설치된 torch 소스를 그대로 읽음.


In [7]:
TORCH = Path(torch.__file__).parent

src = (TORCH / "_dynamo/variables/builder.py").read_text().splitlines()
hit = next(i for i, line in enumerate(src) if "purposely graph breaks on RNN" in line)
print(f"--- torch/_dynamo/variables/builder.py:{hit - 3}-{hit + 1} ---")
print("\n".join(src[hit - 4:hit + 1]))
print()

cfg = (TORCH / "_dynamo/config.py").read_text().splitlines()
hit = next(i for i, line in enumerate(cfg) if line.startswith("allow_rnn"))
print(f"--- torch/_dynamo/config.py:{hit}-{hit + 1} ---")
print("\n".join(cfg[hit - 1:hit + 1]))
print()
print("현재 값:", dynamo.config.allow_rnn)

--- torch/_dynamo/variables/builder.py:1542-1546 ---
        if (
            isinstance(value, (torch.nn.RNN, torch.nn.GRU, torch.nn.LSTM))
            and not config.allow_rnn
        ):
            unimplemented("TorchDynamo purposely graph breaks on RNN, GRU, LSTMs")

--- torch/_dynamo/config.py:373-374 ---
# Disables graph breaking on rnn. YMMV with backends.
allow_rnn = False

현재 값: False


**`nn.RNN` · `nn.GRU` · `nn.LSTM` 셋 다 같은 조건에 걸리고, 기본값은 `False` 임.**

torch 자신이 주석에 `YMMV with backends` 를 달아 둔 **실험적 플래그**임 —
성능이 좋아진다는 보장이 없다는 뜻.


## 5. `allow_rnn=True` 면 LSTM 이 traced 되나 — 최소 예제

BSRNN 전체가 아니라 **LSTM 하나짜리 장난감 모델**로 확인함.
`explain` 은 백엔드를 안 돌리므로 빠르고, 이 절의 관심은 속도가 아니라 **traced 되는가** 임.


In [8]:
lstm_rows = []
for allow in (False, True):
    dynamo.reset()
    dynamo.config.allow_rnn = allow
    toy = torch.nn.LSTM(64, 64, 1, batch_first=True, bidirectional=True).cuda().train()
    xt = torch.randn(4, 50, 64, device="cuda")

    exp = dynamo.explain(lambda t: toy(t)[0])(xt)
    lstm_rows.append({"allow_rnn": allow, "graph_count": exp.graph_count,
                      "graph_break_count": exp.graph_break_count, "op_count": exp.op_count,
                      "판정": "컴파일 안 붙음" if exp.graph_count == 0 else "통째로 traced"})

display(pd.DataFrame(lstm_rows))

,allow_rnn,graph_count,graph_break_count,op_count,판정
0,False,0,-1,0,컴파일 안 붙음
1,True,1,0,4,통째로 traced


**`allow_rnn=False` 일 때 `graph_count` 가 `0` 임 — 컴파일이 아예 안 붙은 것.**

`break_count` 가 `-1` 로 나오는 것은 그래프가 하나도 안 만들어져 셀 것이 없기 때문임.
여기서 중요한 것은 **에러가 나지 않는다는 점** — `compile()` 을 불렀으니 됐겠거니 하고 넘어가기 쉬움.


## 6. `model.compile()` 과 `_orig_mod.` 접두사

`torch.compile(model)` 은 모델을 `OptimizedModule` 로 **감싸므로** `state_dict()` 키에 `_orig_mod.` 가 붙음.
`model.compile()` 은 제자리에서 등록만 하므로 안 붙음 — 이 절이 그것을 확인함.

동시에 **`allow_rnn=True` 로 LSTM 이 든 모델이 실제로 컴파일되어 학습 가능한지**도 같이 봄
(forward 뿐 아니라 **backward 까지** 도는지).


In [9]:
class ToyLSTM(torch.nn.Module):
    """BSRNN 의 ResRNN 과 같은 골격 — LSTM 뒤에 선형 사영."""

    def __init__(self):
        super().__init__()
        self.rnn = torch.nn.LSTM(64, 64, 1, batch_first=True, bidirectional=True)
        self.proj = torch.nn.Linear(128, 64)

    def forward(self, x):
        return self.proj(self.rnn(x)[0])


dynamo.reset()
dynamo.config.allow_rnn = True            # model.compile() 보다 먼저 켜야 함

toy = ToyLSTM().cuda().train()
before = list(toy.state_dict())[:3]
# counters 는 노트북 전체에 걸쳐 누적되고 dynamo.reset() 으로도 안 지워짐 -> 이 셀 몫만 증분으로 셈
g0 = dynamo.utils.counters["stats"]["unique_graphs"]
toy.compile()                             # torch.compile(toy) 가 아님
after = list(toy.state_dict())[:3]

xt = torch.randn(8, 100, 64, device="cuda")
torch.cuda.synchronize()
t0 = time.perf_counter()
toy(xt).pow(2).mean().backward()          # 첫 스텝에서 실제 컴파일이 일어남
torch.cuda.synchronize()
cold = time.perf_counter() - t0

print("compile 전 키:", before)
print("compile 후 키:", after)
print()
display(pd.DataFrame([{
    "_orig_mod. 붙음": any("_orig_mod" in k for k in toy.state_dict()),
    "키 그대로": before == after,
    "이 셀에서 만든 graph": dynamo.utils.counters["stats"]["unique_graphs"] - g0,
    "fwd+bwd": "정상",
    "grad 합": round(toy.rnn.weight_ih_l0.grad.abs().sum().item(), 4),
    "첫 스텝 s": round(cold, 1),
}]))

compile 전 키: ['rnn.weight_ih_l0', 'rnn.weight_hh_l0', 'rnn.bias_ih_l0']
compile 후 키: ['rnn.weight_ih_l0', 'rnn.weight_hh_l0', 'rnn.bias_ih_l0']



,_orig_mod. 붙음,키 그대로,이 셀에서 만든 graph,fwd+bwd,grad 합,첫 스텝 s
0,False,True,1,정상,0.1622,152.0


**`_orig_mod.` 가 안 붙고, graph 1개로 LSTM 이 통째로 컴파일됐으며, 역전파도 돎.**

다만 위 표의 **`첫 스텝 s`** 를 볼 것 — LSTM 1층짜리 장난감인데도 그만큼 듦. 그런 이유는
`allow_rnn=True` 가 cuDNN 의 융합 RNN 커널을 풀어 **Inductor 가 시간축 루프를 직접 짜기** 때문임.
[bsrnn.py](../wesep/models/bsrnn.py) 는 LSTM 이 12개라 콜드 비용이 이보다 훨씬 큼.


## 7. wesep 저장소 쪽 함정 — `strict=False`

컴파일을 켜기 전에 반드시 봐야 하는 곳임. 원문이 근거이므로 그대로 찍음.


In [10]:
ck = (WESEP / "wesep/utils/checkpoint.py").read_text().splitlines()
print("--- wesep/utils/checkpoint.py:63-69 (load_checkpoint) ---")
print("\n".join(ck[62:69]))
print()
print("--- wesep/utils/checkpoint.py:88-93 (save_checkpoint) ---")
print("\n".join(ck[87:93]))

--- wesep/utils/checkpoint.py:63-69 (load_checkpoint) ---
    for model, state in zip(models, model_state):
        if isinstance(model, torch.nn.DataParallel):
            model.module.load_state_dict(state, strict=False)
        elif isinstance(model, torch.nn.parallel.DistributedDataParallel):
            model.module.load_state_dict(state, strict=False)
        else:
            model.load_state_dict(state, strict=False)

--- wesep/utils/checkpoint.py:88-93 (save_checkpoint) ---
    if isinstance(models[0], torch.nn.DataParallel):
        state_dict = [model.module.state_dict() for model in models]
    elif isinstance(models[0], torch.nn.parallel.DistributedDataParallel):
        state_dict = [model.module.state_dict() for model in models]
    else:
        state_dict = [model.state_dict() for model in models]


[save_checkpoint](../wesep/utils/checkpoint.py#L88-L93) 은 DDP 일 때 `model.module.state_dict()` 를 저장함.
`torch.compile(model)` 을 **DDP 로 감싸기 전에** 걸면 `model.module` 이 `OptimizedModule` 이 되어
저장되는 키 전부에 `_orig_mod.` 가 붙음.

문제는 [load_checkpoint](../wesep/utils/checkpoint.py#L63-L69) 의 **`strict=False`** 임 —
키가 **하나도 맞지 않아도 예외를 던지지 않음.**
[run.sh:123-125](../examples/librimix/tse/v2/run.sh#L123-L125) 가 `latest_checkpoint.pt` 를 자동으로 이어받으므로,
컴파일을 켜고 끄는 사이에 재개하면 **학습이 조용히 랜덤 초기화에서 다시 시작됨.**

`model.compile()` 을 쓰면 6절에서 본 대로 접두사가 안 붙어 이 경로 자체가 생기지 않음.


## 정리


In [11]:
display(pd.DataFrame(rows))
display(pd.DataFrame(len_rows))
display(pd.DataFrame(lstm_rows))

,case,first_step_s,ms_per_step,note
0,eager,NaN,476.9,기준
1,compile 기본,97.5,456.3,1.05x
2,compile reduce-overhead,79.7,452.9,1.05x


,enroll_len,step_s,재컴파일
0,598,9.88,예
1,512,33.56,예
2,701,0.48,아니오
3,455,0.47,아니오
4,623,0.47,아니오
5,380,0.47,아니오


,allow_rnn,graph_count,graph_break_count,op_count,판정
0,False,0,-1,0,컴파일 안 붙음
1,True,1,0,4,통째로 traced


| 항목 | 기존 문제점 | 해결방안 | 상태 |
|---|---|---|---|
| BSRNN 컴파일 가능 여부 | — | 됨. graph break 5개 | **확인함** |
| 컴파일 효과 | LSTM 12개가 eager 로 빠져 **1.05×** 에 그침 | `allow_rnn=True` 로 LSTM 까지 넣기 | **1.05× 는 확인함.** `allow_rnn=True` 의 BSRNN 속도는 **미측정** |
| LSTM 컴파일 | 기본값 `allow_rnn=False` 에서 `graph_count = 0` — 에러 없이 조용히 안 붙음 | `torch._dynamo.config.allow_rnn = True` 를 `compile()` **전에** | **확인함** |
| 콜드 컴파일 시간 | 장난감 LSTM 하나도 첫 스텝이 수십 초 (6절 표의 `첫 스텝 s`) | 없음 — `allow_rnn=True` 의 대가임. BSRNN 은 LSTM 12개라 더 큼 | **확인함** |
| `_orig_mod.` 접두사 | `torch.compile(model)` 은 키를 바꿔 기존 체크포인트를 못 읽게 함 | **`model.compile()`** 을 씀 | **확인함** |
| 체크포인트 조용한 실패 | [checkpoint.py:69](../wesep/utils/checkpoint.py#L69) 의 `strict=False` 가 키 전부 불일치를 통과시킴 | `model.compile()` 로 접두사를 안 만들거나, `strict=True` 로 바꿈 | **미조치** — 저장소를 안 고쳤음 |
| CUDA Graphs | `torch.hann_window` 를 CPU 에서 만들어 매 forward `.to(device)` ([bsrnn.py:313](../wesep/models/bsrnn.py#L313) · [:386](../wesep/models/bsrnn.py#L386)) → `reduce-overhead` 가 무력화됨 | 버퍼로 등록해 GPU 에 상주시킴 | **미조치** — 저장소를 안 고쳤음 |
| 컴파일 인자를 어디에 둘까 | — | [utils.py:90](../wesep/utils/utils.py#L90) 이 CLI 를 yaml 위에 덮으므로 `--use_compile true` 도 닿지만,<br>[confs/](../examples/librimix/tse/v2/confs/) 의 yaml 에 두면 [debug.yaml](../examples/librimix/tse/v2/confs/debug.yaml) 덮어쓰기를 그대로 씀 | **미결정** — 사용자 판단 |
| 그래프를 몇 개로 쪼갤까 | SD-FiLM 은 `query_net`·`separation_net` 둘로 나눠 컴파일함 | wesep 쪽 검토 안 함 | **미조사** |
